# ATML PA2 — Task 2: PPO
Use an A100 GPU runtime. Artifacts persist in `MyDrive/PA2`, alongside Task 1. This notebook implements the prescribed experiments and exports numeric results; write your own report and qualitative discussion.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, sys, subprocess, shutil, json, signal

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/PA2')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['PA2_ARTIFACT_ROOT'] = str(DRIVE_ROOT)
os.environ['USE_TF'] = '0'
os.environ['PYTHONUNBUFFERED'] = '1'
REPO = Path('/content/ATML')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/WasifAhmed1917/ATML.git', str(REPO)], check=True)
else:
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO, check=True)
PA2 = REPO / 'PA2'
os.chdir(PA2)
assert (PA2 / 'task2_ppo/run_all.py').exists(), 'Task 2 code must be published to GitHub first.'
print('Code revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())
print('Persistent artifacts:', DRIVE_ROOT)


## Dependencies and live output
Every subprocess streams tqdm and saves a Drive log. Exit codes are checked: a failed command stops the cell. After a disconnected runtime, rerun setup, dependencies, and validation, then the unfinished stage with `--resume`. Keep the same code revision while resuming partial runs.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)

def run(module, *args, artifact_root=None):
    env = {**os.environ, 'PYTHONUNBUFFERED': '1', 'USE_TF': '0'}
    if artifact_root is not None:
        env['PA2_ARTIFACT_ROOT'] = str(artifact_root)
    log_dir = DRIVE_ROOT / 'logs/task2_ppo'
    log_dir.mkdir(parents=True, exist_ok=True)
    label = module.replace('.', '_') + ('_' + args[args.index('--stage')+1] if '--stage' in args else '')
    command = [sys.executable, '-u', '-m', module, *args]
    print('Running:', ' '.join(command), flush=True)
    with (log_dir / (label + '.log')).open('a') as log:
        process = subprocess.Popen(command, cwd=PA2, env=env, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line); log.flush()
            status = process.wait()
        except BaseException:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            process.wait()
            raise
    print('Exit code:', status)
    if status:
        raise subprocess.CalledProcessError(status, command)
run('scripts.check_environment')


## Course assets and objective validation
The downloader uses the pinned course asset revision. Each fork starts from the supplied PPO policy adapter and merged critic, with fresh optimizer states for the continuation. No critic repair or fresh policy initialization is used.

In [ ]:
run('scripts.download_assets')
run('task2_ppo.validate_data')
run('unittest', 'discover', '-s', 'tests', '-p', 'test_task2*.py', '-v')


## GPU smoke test
One update and two evaluation prompts with 16-token caps. Smoke artifacts live in a separate folder and are rejected by full-run exports.

In [ ]:
SMOKE_ROOT = DRIVE_ROOT / 'task2_smoke'
run('task2_ppo.continue_train', '--run-name', 'smoke', '--output', 'outputs/task2_ppo/smoke', '--smoke', '--resume', '--skip-completed', artifact_root=SMOKE_ROOT)
run('task2_ppo.evaluate', '--adapter', 'outputs/task2_ppo/smoke', '--name', 'smoke', '--smoke', '--resume', artifact_root=SMOKE_ROOT)


## Standard: 20-update continuation
Also evaluates the supplied midpoint as a frozen baseline on the same 200 held-out prompts. Training uses up to 512 response tokens; held-out evaluation uses up to 768, as released. Both include EOS and missing-EOS diagnostics.

In [ ]:
run('task2_ppo.run_all', '--stage', 'standard', '--resume')


## Clipping: cached geometry and three eight-update forks
Epsilon = 0.05, 0.20, 0.50. All use the same midpoint, first eight training prompts, generation seeds, 512-token caps, optimizer settings, reward model, and KL coefficient. The cache uses its stored rewards, values, old/reference log probabilities; the midpoint supplies replay probabilities. Advantages normalize once across valid cached tokens.

In [ ]:
run('task2_ppo.run_all', '--stage', 'clipping', '--resume')


## KL pressure: three eight-update forks
Beta_KL = 0, 0.10, 0.20. Each branch starts from the same supplied policy and critic. Maximum generation budgets and update counts match; actual token counts are saved because EOS can terminate responses early.

In [ ]:
run('task2_ppo.run_all', '--stage', 'kl', '--resume')


## Export and plots
Checks all full-run budgets and fixed identities before exporting. `qualitative_review.jsonl` contains all held-out responses, reward scores, and blank fields for your manual review. Existing review notes are preserved.

In [ ]:
run('task2_ppo.run_all', '--stage', 'export', '--resume')


In [ ]:
import pandas as pd
from IPython.display import display, Image
RESULTS = DRIVE_ROOT / 'results/task2_ppo'
for name in ['summary.csv', 'cached_clipping.csv']:
    display(pd.read_csv(RESULTS / name))
print(json.loads((RESULTS / 'completion_checks.json').read_text()))
for path in sorted((RESULTS / 'figures').glob('*.png')):
    display(Image(filename=str(path)))


## Save results
The archive is copied to Drive and downloaded. The standard final policy adapter and critic adapter remain in `MyDrive/PA2/outputs/task2_ppo/standard` for subsequent assignment tasks.

In [ ]:
archive = shutil.make_archive('/content/ATML_PA2_T2_results', 'zip', root_dir=DRIVE_ROOT / 'results', base_dir='task2_ppo')
shutil.copy2(archive, DRIVE_ROOT / Path(archive).name)
files.download(archive)
